In [39]:
import pandas as pd
import numpy as np
import os


def create_label(base_dir):
    data = []
    emotions = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

    for emotion in emotions:
        emotion_dir = os.path.join(base_dir, emotion)
        for img in os.listdir(emotion_dir):
            if img.endswith('.jpg'):  # Add supported image extensions
                img_path = os.path.join(emotion_dir, img)
                data.append([img_path, emotion])

    # Create a DataFrame with two columns: 'image_path' and 'emotion'
    df = pd.DataFrame(data, columns=['image_path', 'emotion'])
    return df

# Load dataset and create DataFrames
dataset = r'D:/Downloads/Emotion detection dataset'
train_set = create_label(os.path.join(dataset, 'train'))
test_set = create_label(os.path.join(dataset, 'test'))

# Save as CSV for future reference
train_set.to_csv('train_labels.csv', index=False)
test_set.to_csv('test_labels.csv', index=False)

# Load the CSV files
train_df = pd.read_csv('train_labels.csv')
test_df = pd.read_csv('test_labels.csv')



In [12]:
!pip install tensorflow
!pip install opencv-python

In [41]:

import cv2
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# Parameters
image_size = (48, 48)  
num_emo = 7  # Number of emotion categories

# Function to preprocess grayscale images
def preprocess_images(df):
    images = []
    labels = []
    
    for _, row in df.iterrows():
        # Load image in grayscale
        img = cv2.imread(row['image_path'], cv2.IMREAD_GRAYSCALE)
        
        # Resize image
        img_resized = cv2.resize(img, image_size)
        
        # Normalize pixel values (0 to 255 -> 0 to 1)
        img_normalized = img_resized / 255.0
        
        # Add a single channel dimension
        images.append(img_normalized[..., np.newaxis])
        labels.append(row['emotion'])

    # Convert images and labels to numpy arrays
    images = np.array(images)
    labels = np.array(labels)

    return images, labels

# Preprocess the data
X_train, y_train = preprocess_images(train_df)
X_test, y_test = preprocess_images(test_df)

# One-hot encode the labels
y_train_encoded = to_categorical(pd.factorize(y_train)[0], num_classes=num_emo)
y_test_encoded = to_categorical(pd.factorize(y_test)[0], num_classes=num_emo)

# Split training data into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train_encoded, test_size=0.2, random_state=42)

In [43]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten, Dense, 
                                     Dropout, BatchNormalization, Activation)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Data Augmentation
datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Build the model
model = Sequential()

# 1st CNN layer
model.add(Conv2D(64, (3, 3), padding='same', activation='relu', input_shape=(48, 48, 1)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

# 2nd CNN layer
model.add(Conv2D(128, (5, 5), activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

# 3rd CNN layer
model.add(Conv2D(512, (3, 3), activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

# 4th CNN layer
model.add(Conv2D(512, (3, 3), activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

# Flatten the layers
model.add(Flatten())

# Fully connected 1st layer
model.add(Dense(256))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.25))

# Fully connected 2nd layer
model.add(Dense(512))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.25))

# Output layer
model.add(Dense(num_emo, activation='softmax'))

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Print the summary
model.summary()


C:\Users\Asus\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_15 (Conv2D)                   │ (None, 48, 48, 64)          │             640 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_17               │ (None, 48, 48, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_12 (MaxPooling2D)      │ (None, 24, 24, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_15 (Dropout)                 │ (None, 24, 24, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_16 (Conv2D)                   │ (None, 24, 24, 128)         │         204,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_18               │ (None, 24, 24, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_13 (MaxPooling2D)      │ (None, 12, 12, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_16 (Dropout)                 │ (None, 12, 12, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_17 (Conv2D)                   │ (None, 12, 12, 512)         │         590,336 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_19               │ (None, 12, 12, 512)         │           2,048 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_14 (MaxPooling2D)      │ (None, 6, 6, 512)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_17 (Dropout)                 │ (None, 6, 6, 512)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_18 (Conv2D)                   │ (None, 6, 6, 512)           │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_20               │ (None, 6, 6, 512)           │           2,048 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_15 (MaxPooling2D)      │ (None, 3, 3, 512)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_18 (Dropout)                 │ (None, 3, 3, 512)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_3 (Flatten)                  │ (None, 4608)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 256)                 │       1,179,904 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_21               │ (None, 256)                 │           1,024 │
│ (BatchNormalization)                 │                             │              

 Total params: 4,478,727 (17.08 MB)

 Trainable params: 4,474,759 (17.07 MB)

 Non-trainable params: 3,968 (15.50 KB)

In [45]:

# Evaluate on test set

model.fit(
    datagen.flow(X_train, y_train, batch_size=128),
    validation_data=(X_val, y_val),
    epochs=50
)

test_loss, test_accuracy = model.evaluate(X_test, y_test_encoded)
print(f"Test accuracy: {test_accuracy * 100:.2f}%")

# Save the model
model.save('enhanced_emotion_detection_v1.2.h5')

Epoch 1/50


C:\Users\Asus\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


180/180 ━━━━━━━━━━━━━━━━━━━━ 187s 1s/step - accuracy: 0.2086 - loss: 2.0322 - val_accuracy: 0.2567 - val_loss: 1.9722
Epoch 2/50
180/180 ━━━━━━━━━━━━━━━━━━━━ 192s 1s/step - accuracy: 0.2390 - loss: 1.8474 - val_accuracy: 0.1738 - val_loss: 1.8949
Epoch 3/50
180/180 ━━━━━━━━━━━━━━━━━━━━ 190s 1s/step - accuracy: 0.2759 - loss: 1.7837 - val_accuracy: 0.1855 - val_loss: 1.8577
Epoch 4/50
180/180 ━━━━━━━━━━━━━━━━━━━━ 191s 1s/step - accuracy: 0.3336 - loss: 1.6777 - val_accuracy: 0.2853 - val_loss: 1.7201
Epoch 5/50
180/180 ━━━━━━━━━━━━━━━━━━━━ 201s 1s/step - accuracy: 0.3859 - loss: 1.5749 - val_accuracy: 0.4547 - val_loss: 1.4222
Epoch 6/50
180/180 ━━━━━━━━━━━━━━━━━━━━ 191s 1s/step - accuracy: 0.4236 - loss: 1.4990 - val_accuracy: 0.4383 - val_loss: 1.4163
Epoch 7/50
180/180 ━━━━━━━━━━━━━━━━━━━━ 192s 1s/step - accuracy: 0.4436 - loss: 1.4392 - val_accuracy: 0.4232 - val_loss: 1.4971
Epoch 8/50
180/180 ━━━━━━━━━━━━━━━━━━━━ 192s 1s/step - accuracy: 0.4672 - loss: 1.3894 - val_accuracy: 0.381

Test accuracy: 62.94%


In [47]:
def predict_emotion(image_path):
 
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    img_resized = cv2.resize(img, image_size)
    img_normalized = img_resized / 255.0
    img_reshaped = img_normalized.reshape(1, 48, 48, 1)

    prediction = model.predict(img_reshaped)
    emotion = np.argmax(prediction)
    
    emotions = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
    return emotions[emotion]

# Example prediction
image_path = r"D:/Downloads/Emotion detection dataset/test/surprise/PublicTest_99446963.jpg"
predicted_emotion = predict_emotion(image_path)
print(f"Predicted emotion: {predicted_emotion}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step
Predicted emotion: surprise


In [51]:
import cv2
import numpy as np
from tensorflow.keras.preprocessing.image import img_to_array

emotion_labels =  ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise'] 

# Initialize the camera
cap = cv2.VideoCapture(0)  # 0 for the default camera

while True:
    # Capture frame-by-frame
    ret, frame = cap.read()
    if not ret:
        break

    # Convert to grayscale and resize for model input
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    face = cv2.resize(gray, (48, 48))
    face = face / 255.0  # Normalize
    face = img_to_array(face)
    face = np.expand_dims(face, axis=0)

    # Predict emotion
    predictions = model.predict(face)
    max_index = np.argmax(predictions[0])
    emotion = emotion_labels[max_index]

    # Display the resulting frame
    cv2.putText(frame, f'Emotion: {emotion}', (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2, cv2.LINE_AA)
    cv2.imshow('Facial Expression Recognition', frame)

    # Break the loop on 'q' key press
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release the camera and close windows
cap.release()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━

In [35]:
print(f"X_train dtype: {X_train.dtype}")
print(f"y_train dtype: {y_train_encoded.dtype}")


X_train dtype: float64
y_train dtype: float64


In [53]:
import pickle

# Assuming `model` is your trained model
with open('enhanced_emotion_detection_v1.2.h5.pkl', 'wb') as f:
    pickle.dump(model, f)